# Collection Operations Report - 数据抽取

> **⚠️ 此版本已废弃，请使用 v2.0**: `01_Data_Extraction_v2.ipynb`

**版本**: v1.0 (已废弃)  
**日期**: 请根据需要修改 `dt_start` 和 `dt_end`

In [1]:
# 参数配置
dt_start = '2026-03-01'  # 开始日期
dt_end = '2026-03-17'    # 结束日期

In [5]:
from get_write_data_from_dataworks import run_sql
import pandas as pd

---

## 1. TL Data - 每日组维度指标

In [9]:
sql=f'''
-- TL Data: 每日组维度（Group × Day）
-- 用于 TL View 的日指标
SELECT
  owner_bucket AS module,
  owner_group AS group_id,
  dt,
  -- 人力
  COUNT(DISTINCT CASE WHEN last_call_hour - first_call_hour >= 8 THEN owner_name END) AS headcount,
  COUNT(DISTINCT owner_name) AS ownercount,
  -- 在手案量
  SUM(owing_case_cnt) AS total_owing_case,
  -- 覆盖
  SUM(comm_times) AS total_comm_times,
  SUM(own_uncomm_case_cnt) AS total_uncomm_case,
  -- 拨打
  SUM(call_times) AS total_calls,
  SUM(call_connect_times) AS total_connect,
  -- 接通率
  CASE WHEN SUM(call_times) > 0 THEN SUM(call_connect_times) / SUM(call_times) END AS connect_rate
FROM phl_anls.tmp_liujun_ana_11_agent_process_daily
WHERE dt >= '{dt_start}'
  AND dt <= '{dt_end}'
  AND is_outs_owner = 0  -- 仅内催
  AND owner_bucket IN ('S0','S1','S2','M1','M2')
GROUP BY
  owner_bucket,
  owner_group,
  dt
'''
tl_data=run_sql(sql)
tl_data.head(5)

log_view_address: http://logview.odps.aliyun.com/logview/?h=https://service.ap-southeast-1-vpc.maxcompute.aliyun-inc.com/api&p=phl_anls&i=20260318091549587gp412848hrs5&token=cG80WUJmdzlRSCtRQXl3cUxyeTVqdE9iVEc0PSxPRFBTX09CTzpwNF8yNjE4MzAxNzM0MTg2MTQ1NDcsMTc3NjQxNzM0OSx7IlN0YXRlbWVudCI6W3siQWN0aW9uIjpbIm9kcHM6UmVhZCJdLCJFZmZlY3QiOiJBbGxvdyIsIlJlc291cmNlIjpbImFjczpvZHBzOio6cHJvamVjdHMvcGhsX2FubHMvaW5zdGFuY2VzLzIwMjYwMzE4MDkxNTQ5NTg3Z3A0MTI4NDhocnM1Il19XSwiVmVyc2lvbiI6IjEifQ==


,module,group_id,dt,headcount,ownercount,total_owing_case,total_comm_times,total_uncomm_case,total_calls,total_connect,connect_rate
0,M1,M1-Large A,2026-03-02,11,12,1741,15986,18,16669,497,0.029816
1,M1,M1-Large A,2026-03-03,12,12,1789,12464,125,12893,388,0.030094
2,M1,M1-Large A,2026-03-04,12,12,1787,14996,50,15504,480,0.030960
3,M1,M1-Large A,2026-03-05,10,12,1759,15846,47,16422,499,0.030386
4,M1,M1-Large A,2026-03-06,12,12,1702,12850,62,13361,399,0.029863


---

## 2. STL Data - 每周模块维度指标

In [10]:
sql=f'''
-- STL Data: 每周模块维度（Module × Week）
-- 用于 STL View 的周指标
SELECT
  CASE WHEN owner_bucket LIKE '%S1%' THEN 'S1' ELSE owner_bucket END AS module,
  -- 周区间：周起始为周六
  CONCAT(
    CASE
      WHEN WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd')) = 0 THEN
        TO_CHAR(DATEADD(TO_DATE(dt, 'yyyy-MM-dd'), -1, 'dd'), 'yyyy-MM-dd')
      ELSE
        TO_CHAR(DATEADD(TO_DATE(dt, 'yyyy-MM-dd'), - (WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd')) + 1), 'dd'), 'yyyy-MM-dd')
    END,
    '-',
    CASE
      WHEN WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd')) = 0 THEN
        TO_CHAR(DATEADD(TO_DATE(dt, 'yyyy-MM-dd'), 5, 'dd'), 'yyyy-MM-dd')
      ELSE
        TO_CHAR(DATEADD(TO_DATE(dt, 'yyyy-MM-dd'), 5 - WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd')), 'dd'), 'yyyy-MM-dd')
    END
  ) AS week,
  -- 人力
  COUNT(DISTINCT CASE WHEN last_call_hour - first_call_hour >= 8 THEN owner_name END) AS headcount,
  -- 在手案量
  SUM(owing_case_cnt) AS total_owing_case,
  -- 覆盖
  SUM(comm_times) AS total_comm_times,
  -- 拨打
  SUM(call_times) AS total_calls,
  SUM(call_connect_times) AS total_connect
FROM phl_anls.tmp_liujun_ana_11_agent_process_daily
WHERE dt >= '{dt_start}'
  AND dt <= '{dt_end}'
  AND is_outs_owner = 0
  AND owner_bucket IN ('S0','S1','S2','M1','M2')
GROUP BY
  CASE WHEN owner_bucket LIKE '%S1%' THEN 'S1' ELSE owner_bucket END,
  CONCAT(
    CASE
      WHEN WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd')) = 0 THEN
        TO_CHAR(DATEADD(TO_DATE(dt, 'yyyy-MM-dd'), -1, 'dd'), 'yyyy-MM-dd')
      ELSE
        TO_CHAR(DATEADD(TO_DATE(dt, 'yyyy-MM-dd'), - (WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd')) + 1), 'dd'), 'yyyy-MM-dd')
    END,
    '-',
    CASE
      WHEN WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd')) = 0 THEN
        TO_CHAR(DATEADD(TO_DATE(dt, 'yyyy-MM-dd'), 5, 'dd'), 'yyyy-MM-dd')
      ELSE
        TO_CHAR(DATEADD(TO_DATE(dt, 'yyyy-MM-dd'), 5 - WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd')), 'dd'), 'yyyy-MM-dd')
    END
  )
'''
stl_data=run_sql(sql)
stl_data.head(5)

log_view_address: http://logview.odps.aliyun.com/logview/?h=https://service.ap-southeast-1-vpc.maxcompute.aliyun-inc.com/api&p=phl_anls&i=20260318091551735gk2npja1zw6&token=cTJWSVViVmNSQ0FpdzA3THcxMWRWZXZINkRrPSxPRFBTX09CTzpwNF8yNjE4MzAxNzM0MTg2MTQ1NDcsMTc3NjQxNzM1MSx7IlN0YXRlbWVudCI6W3siQWN0aW9uIjpbIm9kcHM6UmVhZCJdLCJFZmZlY3QiOiJBbGxvdyIsIlJlc291cmNlIjpbImFjczpvZHBzOio6cHJvamVjdHMvcGhsX2FubHMvaW5zdGFuY2VzLzIwMjYwMzE4MDkxNTUxNzM1Z2sybnBqYTF6dzYiXX1dLCJWZXJzaW9uIjoiMSJ9


,module,week,headcount,total_owing_case,total_comm_times,total_calls,total_connect
0,M1,2026-03-01-2026-03-07,31,32931,231656,235518,6614
1,M1,2026-03-08-2026-03-14,31,32860,252867,257416,7119
2,M1,2026-03-15-2026-03-21,31,13024,83457,84680,2535
3,M2,2026-03-01-2026-03-07,4,16726,27188,27337,924
4,M2,2026-03-08-2026-03-14,4,18010,31101,31212,1048


---

## 3. Agent Performance - Agent日明细

In [13]:
sql=f'''
-- Agent Performance: Agent日明细（Group × Agent × Day）
-- 用于 TL View  drill-down
SELECT
  owner_bucket AS module,
  owner_group AS group_id,
  owner_name AS agent_id,
  dt,
  -- 出勤
  last_call_hour - first_call_hour AS work_hours,
  CASE WHEN last_call_hour - first_call_hour >= 8 THEN 1 ELSE 0 END AS is_full_attendance,
  -- 在手案量
  owing_case_cnt AS owing_case,
  -- 覆盖
  comm_times,
  own_uncomm_case_cnt AS uncomm_case,
  -- 拨打
  call_times,
  call_connect_times AS connect_times,
  -- PTP (Promise to Pay)
  --comm_ptp_cnt AS ptp_cnt,
  -- 接通率
  CASE WHEN call_times > 0 THEN call_connect_times / call_times END AS connect_rate
FROM phl_anls.tmp_liujun_ana_11_agent_process_daily
WHERE dt >= '{dt_start}'
  AND dt <= '{dt_end}'
  AND is_outs_owner = 0
  AND owner_bucket IN ('S0','S1','S2','M1','M2')
ORDER BY
  owner_bucket, owner_group, owner_name, dt
'''
agent_performance=run_sql(sql)
agent_performance.head(5)

log_view_address: http://logview.odps.aliyun.com/logview/?h=https://service.ap-southeast-1-vpc.maxcompute.aliyun-inc.com/api&p=phl_anls&i=202603180916459gkq12848hrs5&token=bzVNVEZEa0pBOURtZW9yeUlwK2dIMUJkU3BrPSxPRFBTX09CTzpwNF8yNjE4MzAxNzM0MTg2MTQ1NDcsMTc3NjQxNzQwNSx7IlN0YXRlbWVudCI6W3siQWN0aW9uIjpbIm9kcHM6UmVhZCJdLCJFZmZlY3QiOiJBbGxvdyIsIlJlc291cmNlIjpbImFjczpvZHBzOio6cHJvamVjdHMvcGhsX2FubHMvaW5zdGFuY2VzLzIwMjYwMzE4MDkxNjQ1OWdrcTEyODQ4aHJzNSJdfV0sIlZlcnNpb24iOiIxIn0=


,module,group_id,agent_id,dt,work_hours,is_full_attendance,owing_case,comm_times,uncomm_case,call_times,connect_times,connect_rate
0,M1,M1-Large A,Ramos02,2026-03-02,9.983333,1,1,4,0,4,0,0.000000
1,M1,M1-Large A,Ramos02,2026-03-02,9.983333,1,1,3,0,3,0,0.000000
2,M1,M1-Large A,Ramos02,2026-03-02,9.983333,1,84,595,0,595,19,0.031933
3,M1,M1-Large A,Ramos02,2026-03-02,9.983333,1,60,741,0,741,18,0.024291
4,M1,M1-Large A,Ramos02,2026-03-03,10.150000,1,1,0,1,0,0,NaN


---

## 4. Group Performance - 组周明细

In [16]:
sql=f'''
-- Group Performance: 组周明细（Module × Group × Week）
-- 用于 STL View drill-down
SELECT
  CASE WHEN owner_bucket LIKE '%S1%' THEN 'S1' ELSE owner_bucket END AS module,
  owner_group AS group_id,
  CONCAT(
    CASE
      WHEN WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd')) = 0 THEN
        TO_CHAR(DATEADD(TO_DATE(dt, 'yyyy-MM-dd'), -1, 'dd'), 'yyyy-MM-dd')
      ELSE
        TO_CHAR(DATEADD(TO_DATE(dt, 'yyyy-MM-dd'), - (WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd')) + 1), 'dd'), 'yyyy-MM-dd')
    END,
    '-',
    CASE
      WHEN WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd')) = 0 THEN
        TO_CHAR(DATEADD(TO_DATE(dt, 'yyyy-MM-dd'), 5, 'dd'), 'yyyy-MM-dd')
      ELSE
        TO_CHAR(DATEADD(TO_DATE(dt, 'yyyy-MM-dd'), 5 - WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd')), 'dd'), 'yyyy-MM-dd')
    END
  ) AS week,
  -- 人力
  COUNT(DISTINCT CASE WHEN last_call_hour - first_call_hour >= 8 THEN owner_name END) AS headcount,
  -- 在手案量
  SUM(owing_case_cnt) AS total_owing_case,
  -- 覆盖
  SUM(comm_times) AS total_comm_times,
  SUM(own_uncomm_case_cnt) AS total_uncomm_case,
  -- 拨打
  SUM(call_times) AS total_calls,
  SUM(call_connect_times) AS total_connect
  -- PTP
  -- SUM(comm_ptp_cnt) AS total_ptp
FROM phl_anls.tmp_liujun_ana_11_agent_process_daily
WHERE dt >= '{dt_start}'
  AND dt <= '{dt_end}'
  AND is_outs_owner = 0
  AND owner_bucket IN ('S0','S1','S2','M1','M2','T2','T4','T5')
GROUP BY
  CASE WHEN owner_bucket LIKE '%S1%' THEN 'S1' ELSE owner_bucket END,
  owner_group,
  CONCAT(
    CASE
      WHEN WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd')) = 0 THEN
        TO_CHAR(DATEADD(TO_DATE(dt, 'yyyy-MM-dd'), -1, 'dd'), 'yyyy-MM-dd')
      ELSE
        TO_CHAR(DATEADD(TO_DATE(dt, 'yyyy-MM-dd'), - (WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd')) + 1), 'dd'), 'yyyy-MM-dd')
    END,
    '-',
    CASE
      WHEN WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd')) = 0 THEN
        TO_CHAR(DATEADD(TO_DATE(dt, 'yyyy-MM-dd'), 5, 'dd'), 'yyyy-MM-dd')
      ELSE
        TO_CHAR(DATEADD(TO_DATE(dt, 'yyyy-MM-dd'), 5 - WEEKDAY(TO_DATE(dt, 'yyyy-MM-dd')), 'dd'), 'yyyy-MM-dd')
    END
  )'''
group_performance=run_sql(sql)
group_performance.head(5)

log_view_address: http://logview.odps.aliyun.com/logview/?h=https://service.ap-southeast-1-vpc.maxcompute.aliyun-inc.com/api&p=phl_anls&i=20260318091806631galopja1zw6&token=UUZqSGlMdTlqbi8wNGhNV1Nqeksxc2pBWldvPSxPRFBTX09CTzpwNF8yNjE4MzAxNzM0MTg2MTQ1NDcsMTc3NjQxNzQ4Nix7IlN0YXRlbWVudCI6W3siQWN0aW9uIjpbIm9kcHM6UmVhZCJdLCJFZmZlY3QiOiJBbGxvdyIsIlJlc291cmNlIjpbImFjczpvZHBzOio6cHJvamVjdHMvcGhsX2FubHMvaW5zdGFuY2VzLzIwMjYwMzE4MDkxODA2NjMxZ2Fsb3BqYTF6dzYiXX1dLCJWZXJzaW9uIjoiMSJ9


,module,group_id,week,headcount,total_owing_case,total_comm_times,total_uncomm_case,total_calls,total_connect
0,M1,M1-Large A,2026-03-01-2026-03-07,12,10478,86514,340,89790,2713
1,M1,M1-Large A,2026-03-08-2026-03-14,12,9718,94505,180,98291,2798
2,M1,M1-Large A,2026-03-15-2026-03-21,12,3907,32151,116,33160,988
3,M1,M1-Large B,2026-03-01-2026-03-07,12,10020,97171,492,97452,2029
4,M1,M1-Large B,2026-03-08-2026-03-14,12,9716,104119,151,104417,2351


---

## 5. Natural Month Repay - 回款数据（日粒度）

In [22]:
sql=f'''
-- Natural Month Repay: 回款数据（日粒度）
-- 用于 Module Daily Trends 和 Risk Analysis
-- 注意：dt 是业务发生日的后一天
SELECT
  dt_biz,
  dt,
  SUBSTR(dt_biz, 1, 7) AS month,
  agent_bucket as module,
  SUM(repay_principal) AS repay_principal,
  SUM(start_owing_principal) AS start_owing_principal
FROM tmp_maoruochen_phl_repay_natural_day_daily
WHERE dt_biz >= '{dt_start}'
  AND dt_biz <= '{dt_end}'
  AND in_or_out = 'inhouse'  -- 仅内催
  AND data_level = '4.组别层级'  -- 按group_name聚合，因此用组别粒度
  AND case_bucket IN ('S0','S1','S2','M1','M2')
GROUP BY
  dt_biz,
  dt,
  SUBSTR(dt_biz, 1, 7),
  agent_bucket'''
module_repay_daily=run_sql(sql)
module_repay_daily.head(5)

log_view_address: http://logview.odps.aliyun.com/logview/?h=https://service.ap-southeast-1-vpc.maxcompute.aliyun-inc.com/api&p=phl_anls&i=20260318092257158g93tzh0qjvk&token=L0c1SGU4NWdORVowZzZOS2RKZ0I1QUhTenNnPSxPRFBTX09CTzpwNF8yNjE4MzAxNzM0MTg2MTQ1NDcsMTc3NjQxNzc3Nyx7IlN0YXRlbWVudCI6W3siQWN0aW9uIjpbIm9kcHM6UmVhZCJdLCJFZmZlY3QiOiJBbGxvdyIsIlJlc291cmNlIjpbImFjczpvZHBzOio6cHJvamVjdHMvcGhsX2FubHMvaW5zdGFuY2VzLzIwMjYwMzE4MDkyMjU3MTU4ZzkzdHpoMHFqdmsiXX1dLCJWZXJzaW9uIjoiMSJ9


,dt_biz,dt,month,module,repay_principal,start_owing_principal
0,2026-03-01,2026-03-02,2026-03,M1_Large,98716,26353719.84
1,2026-03-01,2026-03-02,2026-03,M1_Other,252,122039
2,2026-03-01,2026-03-02,2026-03,M1_Small,15275,2623775.24
3,2026-03-01,2026-03-02,2026-03,M2_Other,16504,25562095.26
4,2026-03-01,2026-03-02,2026-03,S0_Other,9009775.88,19610538.74


---

## 6. Group Level Repay - 组维度回款（日粒度）

In [20]:
sql=f'''
-- Group Level Repay: 组维度回款（日粒度）
-- 用于 Risk Module Groups
SELECT
  dt_biz,
  dt,
  SUBSTR(dt_biz, 1, 7) AS month,
  case_bucket AS module,
  group_name,
  SUM(repay_principal) AS repay_principal,
  SUM(start_owing_principal) AS start_owing_principal
FROM tmp_maoruochen_phl_repay_natural_day_daily
WHERE dt_biz >= '{dt_start}'
  AND dt_biz <= '{dt_end}'
  AND in_or_out = 'inhouse'
  AND data_level = '4.组别层级'  -- 组别粒度
  AND case_bucket IN ('S0','S1','S2','M1','M2','T2','T4','T5')
GROUP BY
  dt_biz,
  dt,
  SUBSTR(dt_biz, 1, 7),
  case_bucket,
  group_name'''
group_repay_daily=run_sql(sql)
group_repay_daily.head(5)

log_view_address: http://logview.odps.aliyun.com/logview/?h=https://service.ap-southeast-1-vpc.maxcompute.aliyun-inc.com/api&p=phl_anls&i=20260318092053998gz9eenlk532&token=ZGdCV0ZGa2hvbkphZDVsRFVnOTM1RExuRjVJPSxPRFBTX09CTzpwNF8yNjE4MzAxNzM0MTg2MTQ1NDcsMTc3NjQxNzY1NCx7IlN0YXRlbWVudCI6W3siQWN0aW9uIjpbIm9kcHM6UmVhZCJdLCJFZmZlY3QiOiJBbGxvdyIsIlJlc291cmNlIjpbImFjczpvZHBzOio6cHJvamVjdHMvcGhsX2FubHMvaW5zdGFuY2VzLzIwMjYwMzE4MDkyMDUzOTk4Z3o5ZWVubGs1MzIiXX1dLCJWZXJzaW9uIjoiMSJ9


,dt_biz,dt,month,module,group_name,repay_principal,start_owing_principal
0,2026-03-01,2026-03-02,2026-03,M1,None,252,122039
1,2026-03-01,2026-03-02,2026-03,M1,M1-Large A,42250,13059492.66
2,2026-03-01,2026-03-02,2026-03,M1,M1-Large B,56466,13294227.18
3,2026-03-01,2026-03-02,2026-03,M1,M1-Small A,15275,2623775.24
4,2026-03-01,2026-03-02,2026-03,M2,M2-A module,16504,25562095.26


---

## 输出说明

请依次运行以上 Cell，将结果下载为 CSV 文件：

| Cell | 输出文件名 | 用途 |
|------|-----------|------|
| 1 | tl_data.csv | TL每日指标 |
| 2 | stl_data.csv | STL每周指标 |
| 3 | agent_performance.csv | Agent明细 |
| 4 | group_performance.csv | 组周明细 |
| 5 | module_repay_daily.csv | 模块日回款 |
| 6 | group_repay_daily.csv | 组日回款 |

**注意**: 目标数据（eom_target）使用本地 CSV 文件：
`d:\97data\eom_target_data_202501_202602.csv`

In [23]:
with pd.ExcelWriter('260318_output_automation.xlsx') as writer:
    tl_data.to_excel(writer, sheet_name='tl_data', startrow=0, startcol=0, index=True)
    stl_data.to_excel(writer, sheet_name='stl_data', startrow=0, startcol=0, index=True)
    
    agent_performance.to_excel(writer, sheet_name='agent_performance', startrow=0, startcol=0, index=True)
    group_performance.to_excel(writer, sheet_name='group_performance', startrow=0, startcol=0, index=True)
    
    module_repay_daily.to_excel(writer, sheet_name='module_repay_daily', startrow=0, startcol=0, index=True)
    group_repay_daily.to_excel(writer, sheet_name='group_repay_daily', startrow=0, startcol=0, index=True)